In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# ============================================================
# TVP-VAR GFEVD -> Connectedness
# ------------------------------------------------------------
# 입력
#   - gfevd_all.npy : shape (T_eff, N, N)
#     row = response variable
#     col = shock source
#
# 출력 (모두 산출물 폴더 하나에 저장)
#   - ./connectedness_output/connectedness_time.csv
#   - ./connectedness_output/connectedness_mean.csv
#   - ./connectedness_output/net_direction.csv
#   - ./connectedness_output/pairwise_net_mean.csv
#   - ./connectedness_output/tci_series.png
#   - ./connectedness_output/net_series.png
# ============================================================

# =========================
# Config
# =========================
BASE_DIR = Path("./")
OUT_DIR = BASE_DIR / "connectedness_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GFEVD_FILE = BASE_DIR / "gfevd_all.npy"

# 날짜를 붙이고 싶으면 아래 후보 파일 중 존재하는 것을 사용
DATE_FILE_CANDIDATES = [
    BASE_DIR / "tvpvar_input_scaled.csv",
    BASE_DIR / "tvpvar_input_transformed.csv",
    BASE_DIR / "tvpvar_raw_level_merged.csv",
]

OUT_TIME = OUT_DIR / "connectedness_time.csv"
OUT_MEAN = OUT_DIR / "connectedness_mean.csv"
OUT_NET = OUT_DIR / "net_direction.csv"
OUT_PAIRWISE = OUT_DIR / "pairwise_net_mean.csv"

OUT_TCI_PNG = OUT_DIR / "tci_series.png"
OUT_NET_PNG = OUT_DIR / "net_series.png"

VAR_NAMES = [
    "SOLVPN",
    "COPPER",
    "GOLD",
    "SILVER",
    "DXY",
    "UST10Y",
    "VIX"
]

ROW_SUM_TOL = 1e-6
FORCE_ROW_NORMALIZE = True

# =========================
# Helper
# =========================
def load_dates(target_len: int):
    """
    후보 파일에서 Date 컬럼을 읽어 마지막 target_len개를 반환.
    길이가 맞지 않으면 뒤에서 맞춘다.
    없으면 None 반환.
    """
    for fp in DATE_FILE_CANDIDATES:
        if fp.exists():
            try:
                df = pd.read_csv(fp)
                if "Date" not in df.columns:
                    continue

                dt = pd.to_datetime(df["Date"], errors="coerce").dropna().reset_index(drop=True)

                if len(dt) < target_len:
                    continue

                # TVP-VAR / GFEVD는 lag 이후 길이일 수 있으므로 뒤에서 맞춤
                dt = dt.iloc[-target_len:].reset_index(drop=True)
                return dt

            except Exception as e:
                print(f"[WARN] Failed to read dates from {fp}: {e}")
                continue

    return None


def row_normalize_theta(theta: np.ndarray) -> np.ndarray:
    """
    각 row 합이 1이 되도록 정규화
    """
    row_sum = theta.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    return theta / row_sum


def compute_connectedness(theta: np.ndarray):
    """
    theta: (N, N), row=response, col=shock source

    반환:
        FROM: row off-diagonal sum
        TO:   col off-diagonal sum
        NET:  TO - FROM
        TCI:  off-diagonal sum / N * 100
    """
    diag = np.diag(theta)

    from_ = theta.sum(axis=1) - diag
    to_ = theta.sum(axis=0) - diag
    net_ = to_ - from_
    tci_ = (theta.sum() - np.trace(theta)) / theta.shape[0] * 100.0

    return from_, to_, net_, tci_


def compute_pairwise_net(theta: np.ndarray):
    """
    Pairwise net spillover matrix
    [i, j] = influence of i on j - influence of j on i
           = theta[j, i] - theta[i, j]
    """
    n = theta.shape[0]
    out = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(n):
            if i == j:
                out[i, j] = 0.0
            else:
                out[i, j] = theta[j, i] - theta[i, j]

    return out


# =========================
# Load
# =========================
if not GFEVD_FILE.exists():
    raise FileNotFoundError(f"Not found: {GFEVD_FILE}")

gfevd_all = np.load(GFEVD_FILE)  # shape: (T_eff, N, N)

if gfevd_all.ndim != 3:
    raise ValueError(f"gfevd_all must be 3D, got shape={gfevd_all.shape}")

T_eff, N, N2 = gfevd_all.shape

if N != N2:
    raise ValueError(f"Last two dims must be square, got shape={gfevd_all.shape}")

if len(VAR_NAMES) != N:
    raise ValueError(f"len(VAR_NAMES)={len(VAR_NAMES)} must match N={N}")

print(f"[INFO] Loaded GFEVD: shape={gfevd_all.shape}")

# =========================
# Check / Normalize
# =========================
gfevd_proc = gfevd_all.copy()

row_sum_check = gfevd_proc.sum(axis=2)  # (T, N)
max_row_err = np.max(np.abs(row_sum_check - 1.0))
print(f"[INFO] Max row-sum error before normalization: {max_row_err:.8f}")

if FORCE_ROW_NORMALIZE and max_row_err > ROW_SUM_TOL:
    print("[INFO] Applying row normalization to all theta_t")
    for t in range(T_eff):
        gfevd_proc[t] = row_normalize_theta(gfevd_proc[t])

row_sum_check_after = gfevd_proc.sum(axis=2)
max_row_err_after = np.max(np.abs(row_sum_check_after - 1.0))
print(f"[INFO] Max row-sum error after normalization: {max_row_err_after:.8f}")

# =========================
# Dates
# =========================
dates = load_dates(T_eff)

# =========================
# Time-varying connectedness
# =========================
records = []
pairwise_net_list = []

for t in range(T_eff):
    theta = gfevd_proc[t]

    from_, to_, net_, tci_ = compute_connectedness(theta)
    pw_net = compute_pairwise_net(theta)
    pairwise_net_list.append(pw_net)

    row = {}
    if dates is not None:
        row["Date"] = dates.iloc[t]
    else:
        row["t"] = t

    for i, name in enumerate(VAR_NAMES):
        row[f"FROM_{name}"] = from_[i] * 100.0
        row[f"TO_{name}"] = to_[i] * 100.0
        row[f"NET_{name}"] = net_[i] * 100.0

    row["TCI"] = tci_
    records.append(row)

df_time = pd.DataFrame(records)

# =========================
# Mean connectedness
# =========================
metric_cols = [c for c in df_time.columns if c not in ["Date", "t"]]
mean_vals = df_time[metric_cols].mean(axis=0)

rows_mean = []
for name in VAR_NAMES:
    rows_mean.append({
        "Variable": name,
        "FROM_mean": mean_vals[f"FROM_{name}"],
        "TO_mean": mean_vals[f"TO_{name}"],
        "NET_mean": mean_vals[f"NET_{name}"],
        "TCI_mean": np.nan
    })

rows_mean.append({
    "Variable": "SYSTEM",
    "FROM_mean": np.nan,
    "TO_mean": np.nan,
    "NET_mean": np.nan,
    "TCI_mean": mean_vals["TCI"]
})

df_mean = pd.DataFrame(rows_mean)

# =========================
# Net direction only
# =========================
df_net = df_mean[["Variable", "NET_mean"]].copy()
df_net = df_net[df_net["Variable"] != "SYSTEM"].sort_values("NET_mean", ascending=False).reset_index(drop=True)

# =========================
# Pairwise net mean
# =========================
pairwise_net_all = np.stack(pairwise_net_list, axis=0)   # (T, N, N)
pairwise_net_mean = pairwise_net_all.mean(axis=0) * 100.0
df_pairwise = pd.DataFrame(pairwise_net_mean, index=VAR_NAMES, columns=VAR_NAMES)

# =========================
# Save CSV
# =========================
df_time.to_csv(OUT_TIME, index=False)
df_mean.to_csv(OUT_MEAN, index=False)
df_net.to_csv(OUT_NET, index=False)
df_pairwise.to_csv(OUT_PAIRWISE, index=True)

print("[INFO] Saved CSV files:")
print(f"  - {OUT_TIME}")
print(f"  - {OUT_MEAN}")
print(f"  - {OUT_NET}")
print(f"  - {OUT_PAIRWISE}")

# =========================
# Plot 1: TCI series
# =========================
plt.figure(figsize=(12, 4))

if "Date" in df_time.columns:
    x = pd.to_datetime(df_time["Date"])
    x_label = "Date"
else:
    x = df_time["t"]
    x_label = "t"

plt.plot(x, df_time["TCI"], linewidth=1.5)
plt.title("TCI Time Series")
plt.xlabel(x_label)
plt.ylabel("TCI (%)")
plt.tight_layout()
plt.savefig(OUT_TCI_PNG, dpi=300, bbox_inches="tight")
plt.close()

# =========================
# Plot 2: NET series
# =========================
plt.figure(figsize=(14, 6))
for name in VAR_NAMES:
    plt.plot(x, df_time[f"NET_{name}"], label=name, linewidth=1.2)

plt.axhline(0.0, linestyle="--", linewidth=1.0)
plt.title("NET Spillover Time Series")
plt.xlabel(x_label)
plt.ylabel("NET (%)")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_NET_PNG, dpi=300, bbox_inches="tight")
plt.close()

print("[INFO] Saved plot files:")
print(f"  - {OUT_TCI_PNG}")
print(f"  - {OUT_NET_PNG}")

# =========================
# Print summary
# =========================
print("\n[INFO] Average NET ranking")
print(df_net.to_string(index=False))

print("\n[INFO] Average TCI")
print(f"TCI_mean = {df_time['TCI'].mean():.4f}%")

[INFO] Loaded GFEVD: shape=(1035, 7, 7)
[INFO] Max row-sum error before normalization: 0.00000000
[INFO] Max row-sum error after normalization: 0.00000000
[INFO] Saved CSV files:
  - connectedness_output\connectedness_time.csv
  - connectedness_output\connectedness_mean.csv
  - connectedness_output\net_direction.csv
  - connectedness_output\pairwise_net_mean.csv
[INFO] Saved plot files:
  - connectedness_output\tci_series.png
  - connectedness_output\net_series.png

[INFO] Average NET ranking
Variable   NET_mean
  SILVER   6.476445
    GOLD   6.075409
  SOLVPN   4.628936
     VIX  -1.451217
  COPPER  -1.479075
     DXY  -3.956118
  UST10Y -10.294380

[INFO] Average TCI
TCI_mean = 52.6302%
